<a href="https://colab.research.google.com/github/INNORH/FlyRank-Internship/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/INNORH/FlyRank-Internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [ ]:
I'm picking this lane because Week 1 and Week 2 already pointed me here without me planning it.
 In Week 1 I found that search_volume barely predicts real traffic (correlation ~0.001 with impressions) and that page length isn't the lever separating declining pages from growing ones — both of those rule out the "obvious" fixes an editor might reach for first.
 In Week 2 I found that a hand-written stale+visible rule looks great in-sample but drops to near coin-flip (Precision@20 = 0.50) on held-out clients, while even a shallow, readable decision tree modestly beats it out-of-sample.
  That's the exact shape of problem this lane is built for: there's real signal in the data, but it's not simple enough for one if-statement, and a transparent baseline already exists to be beaten honestly.
  I'd rather spend 7 weeks building a ranked review queue I can defend than a clustering exercise or a sparse AI-referral study, given how much groundwork I've already laid here.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [ ]:
Decision this improves: which pages should a content editor review first this week, out of a client's full inventory, when the editor only has time to look at a handful.

Who acts on it: a FlyRank content editor or account strategist with limited review capacity — they open the ranked queue and work top-down.

The action taken: the editor opens the top-ranked pages, checks the reason codes (why the page was flagged — stale, declining, thin, page-one decay, low CTR, low engagement), and decides whether to refresh, expand, protect, prune, or just monitor the page.

Cost of a wrong call, both directions:

False positive (page flagged as a priority but it wasn't really at risk): wastes an editor's limited review time on a page that didn't need attention — the true cost is opportunity cost, since every hour spent here is an hour not spent on a page that really was declining.
False negative (a genuinely declining, high-traffic page never surfaces in the top of the queue): the page keeps losing visibility unnoticed until it's much further gone, which is the more expensive failure mode since it can mean lost client traffic and revenue over an extended period before anyone catches it.

Because false negatives are costlier than false positives here, I'll care about recall at the top of the queue, not just precision — missing a real decliner is worse than reviewing one healthy page that turned out fine

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{df.shape[0]:,} pages, {df['client_id'].nunique()} clients\n")

# Number 1 -- scale of the problem: how much of the inventory is currently declining?
declining_rate = df["trend_direction"].str.lower().eq("down").mean()
declining_n = df["trend_direction"].str.lower().eq("down").sum()
print(f"1) {declining_n:,} of {df.shape[0]:,} pages ({declining_rate*100:.1f}%) are currently trending down.")
print("   -> This isn't a rare-event problem. Over half the inventory needs some kind of triage,")
print("   which is exactly why a ranked queue (not a manual scan) matters.\n")

# Number 2 -- the actionable, high-value subset: visible AND declining
visible = df["impressions_90d"] >= 500
visible_declining = visible & df["trend_direction"].str.lower().eq("down")
print(f"2) {visible_declining.sum():,} pages ({visible_declining.mean()*100:.1f}% of all pages) are BOTH visible")
print(f"   (>=500 impressions/90d) AND declining -- these are the pages where a wrong call is expensive,")
print(f"   because real traffic is on the line right now. They touch {df.loc[visible_declining,'client_id'].nunique()} of")
print(f"   {df['client_id'].nunique()} clients, so this isn't isolated to one account.\n")

# Number 3 -- evidence a learned rank beats a hand rule (verified pipeline result, outputs/model_results.json)
import json
res = json.load(open("outputs/model_results.json")) if __import__("os").path.exists("outputs/model_results.json") else None
if res:
    base_p50 = res["baseline"]["baseline_precision_at_50"]
    rf_p50 = res["models"]["random_forest"]["precision_at_50"]
else:
    # verified numbers from docs/ml-intern-dataset-and-lane-guide.md Section 5 if pipeline hasn't been rerun
    base_p50, rf_p50 = 0.240, 0.740
print(f"3) On the starter pipeline's client-holdout evaluation, the hand-written baseline rule scores")
print(f"   Precision@50 = {base_p50:.3f}, while a random forest scores Precision@50 = {rf_p50:.3f} --")
print(f"   roughly {rf_p50/base_p50:.1f}x as many correct picks in the top 50. That's the gap this lane exists to widen and validate honestly.")
30,000 pages, 32 clients

1) 16,262 of 30,000 pages (54.2%) are currently trending down.
   -> This isn't a rare-event problem. Over half the inventory needs some kind of triage,
   which is exactly why a ranked queue (not a manual scan) matters.

2) 9,961 pages (33.2% of all pages) are BOTH visible
   (>=500 impressions/90d) AND declining -- these are the pages where a wrong call is expensive,
   because real traffic is on the line right now. They touch 27 of
   32 clients, so this isn't isolated to one account.

3) On the starter pipeline's client-holdout evaluation, the hand-written baseline rule scores
   Precision@50 = 0.240, while a random forest scores Precision@50 = 0.740 --
   roughly 3.1x as many correct picks in the top 50. That's the gap this lane exists to widen and validate honestly.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
What this work WILL be able to say:

Observed: e.g. "pages meeting these criteria were, in this dataset, more likely to carry the down trend label than the inventory average."
Directional: e.g. "a learned ranking captured more true decliners in its top 50 than the hand-written rule did, on held-out clients."
Decision-support: the ranked queue is a prioritization aid for a human reviewer with limited time — it points at where to look first, backed by reason codes the reviewer can inspect and override.
What this work will NEVER say:

No causal proof. I can't claim that refreshing a page causes it to recover — that would require an actual experiment (e.g. before/after refresh with a control group), which this data doesn't give me. At most I can say a page was flagged, and note if it later shows a different trend — never that the flag caused the outcome.
No "predicting Google." I'm not claiming to have modeled Google's ranking algorithm, discovered a ranking factor, or predicted search engine behavior. I'm working from observable client-side signals (impressions, clicks, position, engagement) that were already measured, not reverse-engineering search infrastructure.
No guarantee for any individual page. A page landing high in the queue is evidence worth a look, not a certainty that it's actually declining or that fixing it will help — Week 2 showed my own hand rule's in-sample precision (0.95 at the top) collapsed to 0.50 on unseen clients, so I know firsthand how easily an overconfident-looking result can be an illusion of having already seen the data.
Precision, not perfection. Even my best out-of-sample number so far (Precision@50 = 0.74 from the verified pipeline run) means roughly 1 in 4 top-50 picks won't actually be a real decliner — that's a real, quantified error rate I'll keep surfacing, not hide behind a single headline metric.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.